# Cohort-Level Benchmarks

This notebook shows, in executable code, how the public cohort-level benchmark numbers are computed.

It intentionally does **not** just load the released metric summaries. Instead it:

1. Loads the public 44-row score/feature artifacts.
2. Reimplements the metric functions used by the release.
3. Recomputes Gaia, Atlas, DepMap, and formula-control metrics from row-level data.
4. Checks the recomputed values against the checked-in metric CSVs.
5. Generates the cohort figures under `artifacts/notebook_figures/`.

Raw Gaia checkpoints, per-cell inference outputs, raw Atlas, and raw DepMap matrices are external to this public repo. For Atlas and DepMap, this notebook recomputes from the released row-level prediction/feature artifacts and includes the exact external-source regeneration command at the end.


In [ ]:
# Standard-library imports only. The notebook is intentionally self-contained.
# We do not import the package-level reproduce_* helpers for the calculations below.
from pathlib import Path
import csv
import html
import json
import math
import os
import random
import subprocess
import sys

# IPython display is optional. The notebook still runs as plain Python.
try:
    from IPython.display import SVG, Markdown, display
except Exception:
    SVG = Markdown = None
    def display(value):
        print(value)

# Find the repository root from wherever Jupyter launched this notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the open-benchmarks repo")
    ROOT = ROOT.parent

# Generated figures are reproducible outputs and are intentionally gitignored.
FIG_DIR = ROOT / "artifacts/notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Public cohort artifact locations used by this notebook.
GAIA_DIR = ROOT / "cohort-level-bench/model_scores/gaia"
BASELINE_DIR = ROOT / "cohort-level-bench/baseline/results"
GAIA_ROWS_CSV = GAIA_DIR / "gaia_44_strict_orr_model_scores.csv"
GAIA_METRICS_CSV = GAIA_DIR / "gaia_metrics.csv"
GAIA_BY_DISEASE_CSV = GAIA_DIR / "gaia_by_disease_metrics.csv"
ATLAS_ROWS_CSV = BASELINE_DIR / "atlas_orr_predictions.csv"
ATLAS_METRICS_CSV = BASELINE_DIR / "atlas_orr_metrics.csv"
DEPMAP_ROWS_CSV = BASELINE_DIR / "depmap_orr_features.csv"
DEPMAP_METRICS_CSV = BASELINE_DIR / "depmap_orr_metrics.csv"

# Public cohort formula-control artifacts.
# These controls keep the released Gaia score formula and metric path fixed, then
# corrupt only the ORR label alignment globally or within disease.
COHORT_FORMULA_CONTROL_DIR = ROOT / "cohort-level-bench/baseline/formula_controls_20260605"
COHORT_FORMULA_REAL_CSV = COHORT_FORMULA_CONTROL_DIR / "cohort_formula_control_real_metrics.csv"
COHORT_FORMULA_METRICS_CSV = COHORT_FORMULA_CONTROL_DIR / "cohort_formula_control_metrics.csv"
COHORT_FORMULA_SUMMARY_CSV = COHORT_FORMULA_CONTROL_DIR / "cohort_formula_control_summary.csv"

print(f"Repository root: {ROOT}")
print(f"Figures will be written to: {FIG_DIR.relative_to(ROOT)}")


## Shared CSV and Metric Code

These functions are the benchmark machinery. They are written out here so a reader can see exactly how Pearson, Spearman, AUC, MAE, and the disease-median AUC are computed.

In [ ]:
# -----------------------------------------------------------------------------
# Shared metric helpers for cohort-level benchmarks
# -----------------------------------------------------------------------------
# This cell is the benchmark engine used below. It intentionally redefines the
# metrics in the notebook instead of importing the package-level wrappers, so a
# reviewer can see exactly how every headline number is computed from CSV rows.
#
# Important cohort convention:
#   observed_orr_pct = clinical ORR label in percentage points
#   score_col        = model or baseline score being evaluated
# -----------------------------------------------------------------------------

def read_csv_rows(path):
    """Read a CSV file as a list of dictionaries, preserving all public columns."""
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def finite_float(value, default=math.nan):
    """Convert CSV strings to floats and return NaN for missing/non-finite values."""
    try:
        out = float(value)
    except (TypeError, ValueError):
        return default
    return out if math.isfinite(out) else default


def finite_pairs(xs, ys):
    """Keep only paired finite values before computing a metric."""
    pairs = []
    for x_raw, y_raw in zip(xs, ys):
        x = finite_float(x_raw)
        y = finite_float(y_raw)
        if math.isfinite(x) and math.isfinite(y):
            pairs.append((x, y))
    return pairs


def mean(values):
    """Mean over finite numeric values."""
    clean = [finite_float(value) for value in values]
    clean = [value for value in clean if math.isfinite(value)]
    return sum(clean) / len(clean) if clean else math.nan


def quantile(values, q):
    """Linear-interpolated quantile over finite numeric values."""
    clean = sorted(finite_float(value) for value in values)
    clean = [value for value in clean if math.isfinite(value)]
    if not clean:
        return math.nan
    if len(clean) == 1:
        return clean[0]
    pos = q * (len(clean) - 1)
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return clean[lo]
    frac = pos - lo
    return clean[lo] * (1.0 - frac) + clean[hi] * frac


def pearson(xs, ys):
    """Ordinary Pearson correlation on finite paired values."""
    pairs = finite_pairs(xs, ys)
    if len(pairs) < 2:
        return math.nan
    x_values = [x for x, _ in pairs]
    y_values = [y for _, y in pairs]
    x_mean = mean(x_values)
    y_mean = mean(y_values)
    x_ss = sum((x - x_mean) ** 2 for x in x_values)
    y_ss = sum((y - y_mean) ** 2 for y in y_values)
    if x_ss <= 0.0 or y_ss <= 0.0:
        return math.nan
    cov = sum((x - x_mean) * (y - y_mean) for x, y in pairs)
    return cov / math.sqrt(x_ss * y_ss)


def rankdata(values):
    """Average ranks, 1-indexed, with ties handled like scipy.stats.rankdata."""
    order = sorted(range(len(values)), key=lambda idx: values[idx])
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start
        while end + 1 < len(order) and values[order[end + 1]] == values[order[start]]:
            end += 1
        rank = (start + 1 + end + 1) / 2.0
        for pos in range(start, end + 1):
            ranks[order[pos]] = rank
        start = end + 1
    return ranks


def spearman(xs, ys):
    """Spearman rho as Pearson correlation of average ranks."""
    pairs = finite_pairs(xs, ys)
    if len(pairs) < 2:
        return math.nan
    x_ranks = rankdata([x for x, _ in pairs])
    y_ranks = rankdata([y for _, y in pairs])
    return pearson(x_ranks, y_ranks)


def roc_auc(labels, scores, positive=1):
    """Pairwise AUC: probability a positive row scores above a negative row."""
    pairs = []
    for label_raw, score_raw in zip(labels, scores):
        score = finite_float(score_raw)
        if not math.isfinite(score):
            continue
        try:
            label = int(float(label_raw))
        except (TypeError, ValueError):
            continue
        pairs.append((label, score))
    positives = [score for label, score in pairs if label == positive]
    negatives = [score for label, score in pairs if label != positive]
    if not positives or not negatives:
        return math.nan
    wins = 0.0
    for positive_score in positives:
        for negative_score in negatives:
            if positive_score > negative_score:
                wins += 1.0
            elif positive_score == negative_score:
                wins += 0.5
    return wins / (len(positives) * len(negatives))


def group_rows(rows, key):
    """Group rows by a named CSV column."""
    grouped = {}
    for row in rows:
        grouped.setdefault(str(row.get(key, "")), []).append(row)
    return grouped


def auc_above_disease_median(rows, value_col, score_col, group_col="cohort"):
    """Within each disease, label rows above disease-median observed ORR and compute AUC."""
    medians = {}
    for disease, disease_rows in group_rows(rows, group_col).items():
        values = [finite_float(row.get(value_col)) for row in disease_rows]
        values = [value for value in values if math.isfinite(value)]
        if values:
            medians[disease] = quantile(values, 0.50)
    labels = []
    scores = []
    for row in rows:
        disease = str(row.get(group_col, ""))
        observed = finite_float(row.get(value_col))
        score = finite_float(row.get(score_col))
        if disease in medians and math.isfinite(observed) and math.isfinite(score):
            labels.append(int(observed > medians[disease]))
            scores.append(score)
    return roc_auc(labels, scores)


def pairwise_within_disease_correlation(rows, score_col, method="pearson"):
    """Compare within-disease pairwise score deltas against ORR deltas."""
    pair_rows = []
    for disease, disease_rows in group_rows(rows, "cohort").items():
        valid = [
            row for row in disease_rows
            if math.isfinite(finite_float(row.get(score_col)))
            and math.isfinite(finite_float(row.get("observed_orr_pct")))
        ]
        for left_idx, left in enumerate(valid):
            for right in valid[left_idx + 1:]:
                pair_rows.append({
                    "score_delta": finite_float(left[score_col]) - finite_float(right[score_col]),
                    "orr_delta": finite_float(left["observed_orr_pct"]) - finite_float(right["observed_orr_pct"]),
                })
    corr = spearman if method == "spearman" else pearson
    return corr([row["score_delta"] for row in pair_rows], [row["orr_delta"] for row in pair_rows]), len(pair_rows)


def cohort_metric_row(rows, score_col, is_orr_scale=True):
    """Compute the public cohort metric row for a model or baseline score."""
    scored = [
        row for row in rows
        if math.isfinite(finite_float(row.get(score_col)))
        and math.isfinite(finite_float(row.get("observed_orr_pct")))
    ]
    observed = [finite_float(row["observed_orr_pct"]) for row in scored]
    scores = [finite_float(row[score_col]) for row in scored]
    errors = [score - label for score, label in zip(scores, observed)]
    pair_pearson, n_pairs = pairwise_within_disease_correlation(scored, score_col, "pearson")
    pair_spearman, _ = pairwise_within_disease_correlation(scored, score_col, "spearman")
    return {
        "label": score_col,
        "score_col": score_col,
        "n_rows": len(scored),
        "pearson": pearson(observed, scores),
        "spearman": spearman(observed, scores),
        "within_disease_pair_pearson": pair_pearson,
        "within_disease_pair_spearman": pair_spearman,
        "within_disease_pairs": n_pairs,
        "auc_above_disease_median": auc_above_disease_median(scored, "observed_orr_pct", score_col),
        "mae_orr_pct": mean(abs(error) for error in errors) if is_orr_scale else math.nan,
        "rmse_orr_pct": math.sqrt(mean(error * error for error in errors)) if is_orr_scale else math.nan,
        "mean_prediction": mean(scores),
        "mean_observed_orr": mean(observed),
    }


def assert_close(actual, expected, label, tol=1e-9):
    """Fail loudly if a recomputed value drifts from the checked-in release value."""
    actual_value = finite_float(actual)
    expected_value = finite_float(expected)
    if math.isnan(actual_value) and math.isnan(expected_value):
        return
    if not math.isclose(actual_value, expected_value, rel_tol=tol, abs_tol=tol):
        raise AssertionError(f"{label}: actual={actual_value} expected={expected_value}")


def compare_to_metric_csv(metric, metric_csv, score_col):
    """Compare recomputed metrics against the public metric CSV for auditability."""
    expected_rows = read_csv_rows(metric_csv)
    expected = next(row for row in expected_rows if row.get("score_col") == score_col)
    for key in [
        "n_rows", "pearson", "spearman", "within_disease_pair_pearson",
        "within_disease_pair_spearman", "within_disease_pairs",
        "auc_above_disease_median", "mae_orr_pct", "rmse_orr_pct",
        "mean_prediction", "mean_observed_orr",
    ]:
        if key in expected:
            assert_close(metric[key], expected[key], f"{score_col}.{key}")
    return expected

print("Metric functions loaded.")

## Gaia Strict-44 Cohort ORR

Gaia is evaluated directly from the 44-row public score table. The score column is `gaia_predicted_orr_pct`, and the label is `observed_orr_pct`/`orr_pct`.

In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 1: Gaia strict-44 cohort ORR
# -----------------------------------------------------------------------------
# Input artifact:
#   cohort-level-bench/model_scores/gaia/gaia_44_strict_orr_model_scores.csv
#
# Label column:
#   observed_orr_pct / orr_pct
#
# Score column:
#   gaia_predicted_orr_pct
#
# What this cell does:
#   1. Read the 44 public cohort-drug rows.
#   2. Compute Pearson, Spearman, MAE, RMSE, and AUC above disease median.
#   3. Recompute by-disease metrics from the same rows.
#   4. Assert that all recomputed values match the checked-in metric CSVs.
# -----------------------------------------------------------------------------

# Load the public Gaia score rows.
gaia_rows = read_csv_rows(GAIA_ROWS_CSV)

# Normalize observed ORR into the same column name used by the metric code.
# The CSV includes both `orr_pct` and `observed_orr_pct`; this keeps the code explicit.
for row in gaia_rows:
    row["observed_orr_pct"] = row.get("observed_orr_pct") or row.get("orr_pct")

# Compute the public metrics from row-level values, not from the summary file.
gaia_metric = cohort_metric_row(gaia_rows, "gaia_predicted_orr_pct", is_orr_scale=True)

# Check against the checked-in public metric CSV. This validates exact reproduction.
compare_to_metric_csv(gaia_metric, GAIA_METRICS_CSV, "gaia_predicted_orr_pct")

# Compute by-disease metrics the same way, using only rows in that disease.
gaia_by_disease = []
for disease, disease_rows in sorted(group_rows(gaia_rows, "cohort").items()):
    metric = cohort_metric_row(disease_rows, "gaia_predicted_orr_pct", is_orr_scale=True)
    gaia_by_disease.append({
        "cohort": disease,
        "n_orr": metric["n_rows"],
        "pearson": metric["pearson"],
        "spearman": metric["spearman"],
        "mae_orr_pct": metric["mae_orr_pct"],
        "mean_prediction": metric["mean_prediction"],
        "mean_observed_orr": metric["mean_observed_orr"],
    })

# Validate by-disease rows against the checked-in by-disease metric artifact.
expected_by_disease = {row["cohort"]: row for row in read_csv_rows(GAIA_BY_DISEASE_CSV)}
for row in gaia_by_disease:
    expected = expected_by_disease[row["cohort"]]
    for key in ["n_orr", "pearson", "spearman", "mae_orr_pct", "mean_prediction", "mean_observed_orr"]:
        assert_close(row[key], expected[key], f"gaia_by_disease.{row['cohort']}.{key}")

print(json.dumps(gaia_metric, indent=2, sort_keys=True))

## Atlas Fixed `k=8` ORR Prior

The full Atlas baseline is generated from an external raw Atlas trial-arm CSV. The public repo includes the resulting 44-row Atlas prediction table so the released correlation can be recomputed without private/raw source data.

This cell recomputes the primary `atlas_mono_disease_therapy_shrink_k8` score from the columns in the public prediction table:

```text
shrunk ORR = n/(n+8) * disease_therapy_ORR + 8/(n+8) * global_therapy_ORR
```

If either shrinkage cell is missing, the baseline falls back to the precomputed all-comer therapy-first hierarchy score. Gaia scores and tumor biology are not used.

In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 2: Atlas fixed-k=8 historical ORR prior
# -----------------------------------------------------------------------------
# Input artifact:
#   cohort-level-bench/baseline/results/atlas_orr_predictions.csv
#
# This public artifact is the 44-row output of the raw Atlas baseline script.
# It contains enough support columns to recompute the fixed-k=8 prior without
# loading the raw Atlas trial-arm table.
#
# Label column:
#   observed_orr_pct
#
# Score formula:
#   n/(n+8) * disease_therapy_ORR + 8/(n+8) * global_therapy_ORR
#
# Overfit boundary:
#   The target ORR label is used only after score computation for evaluation.
#   Gaia scores, tumor biology, and exact target-drug Atlas arms are not inputs.
# -----------------------------------------------------------------------------

# Load the public Atlas row-level prediction artifact.
atlas_rows = read_csv_rows(ATLAS_ROWS_CSV)
ATLAS_SCORE = "atlas_mono_disease_therapy_shrink_k8"
ATLAS_K = 8.0

for row in atlas_rows:
    # These two Atlas support cells are generated by the raw-Atlas baseline script
    # after exact target-drug arms have been excluded.
    disease_therapy_orr = finite_float(row.get("atlas_mono_allcomer_prior_disease_therapy_orr"))
    disease_therapy_n = finite_float(row.get("atlas_mono_allcomer_prior_disease_therapy_n_arms"), 0.0)
    global_therapy_orr = finite_float(row.get("atlas_mono_allcomer_prior_global_therapy_orr"))

    # This is the deterministic fallback used when a shrinkage cell is unavailable.
    hierarchy_fallback = finite_float(
        row.get("atlas_mono_allcomer_hierarchical_prior_therapy_first_orr")
    )

    # Recompute the exact released score from the public support columns.
    if math.isfinite(disease_therapy_orr) and math.isfinite(global_therapy_orr):
        support_weight = max(0.0, disease_therapy_n) / (max(0.0, disease_therapy_n) + ATLAS_K)
        recomputed = support_weight * disease_therapy_orr + (1.0 - support_weight) * global_therapy_orr
    else:
        recomputed = hierarchy_fallback

    # Keep the recomputed value separate so we can verify that it matches the released score.
    row["atlas_recomputed_fixed_k8"] = recomputed
    assert_close(recomputed, row[ATLAS_SCORE], f"atlas score {row.get('surface_pair_id')}")

# Compute release metrics from the recomputed Atlas score.
atlas_metric = cohort_metric_row(atlas_rows, "atlas_recomputed_fixed_k8", is_orr_scale=True)

# The checked-in metrics use the released score column name, so compare values key by key.
expected_atlas = read_csv_rows(ATLAS_METRICS_CSV)[0]
for key in [
    "n_rows", "pearson", "spearman", "within_disease_pair_pearson",
    "within_disease_pair_spearman", "within_disease_pairs",
    "auc_above_disease_median", "mae_orr_pct", "rmse_orr_pct",
    "mean_prediction", "mean_observed_orr",
]:
    assert_close(atlas_metric[key], expected_atlas[key], f"atlas.{key}")

print(json.dumps(atlas_metric, indent=2, sort_keys=True))

## DepMap Lineage-Sensitivity Baseline

The full DepMap feature build is generated from external DepMap drug matrices and `Model.csv`. The public repo includes the resulting 44-row feature table.

The primary score is:

```text
depmap_lineage_sensitivity_rank = 1 - mean(percentile_rank(AUC))
```

Higher means matched-lineage cell lines are more sensitive. Here we recompute the released metrics from the public feature rows.

In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 3: DepMap lineage-sensitivity baseline
# -----------------------------------------------------------------------------
# Input artifact:
#   cohort-level-bench/baseline/results/depmap_orr_features.csv
#
# This public artifact is the 44-row output of the raw DepMap baseline script.
# It contains the DepMap-matched feature rows; raw matrices remain external.
#
# Label column:
#   observed_orr_pct
#
# Primary score column:
#   depmap_lineage_sensitivity_rank
# -----------------------------------------------------------------------------

# Load the public 44-row DepMap feature table.
depmap_rows = read_csv_rows(DEPMAP_ROWS_CSV)
DEPMAP_PRIMARY = "depmap_lineage_sensitivity_rank"

# The feature table already contains the row-level sensitivity score generated
# from DepMap matrices. We recompute the benchmark metrics from those rows.
depmap_metric = cohort_metric_row(depmap_rows, DEPMAP_PRIMARY, is_orr_scale=False)

# Validate against the checked-in metric CSV.
compare_to_metric_csv(depmap_metric, DEPMAP_METRICS_CSV, DEPMAP_PRIMARY)

# Also recompute the two non-primary sensitivity orientations reported in the CSV.
depmap_secondary_metrics = []
for score_col in ["depmap_lineage_auc_sensitivity", "depmap_global_auc_sensitivity"]:
    metric = cohort_metric_row(depmap_rows, score_col, is_orr_scale=False)
    compare_to_metric_csv(metric, DEPMAP_METRICS_CSV, score_col)
    depmap_secondary_metrics.append(metric)

print(json.dumps(depmap_metric, indent=2, sort_keys=True))

## Cohort Formula Controls

The released 44-row cohort artifact contains the final Gaia cohort score, not upstream per-patient axis values. The exact public control therefore keeps `gaia_predicted_orr_pct` and the public metric path fixed, then corrupts only the label alignment.

Two deterministic null families are recomputed here:

- `label_shuffle_global`: shuffle `observed_orr_pct` across all 44 rows.
- `label_shuffle_within_disease`: shuffle `observed_orr_pct` within each disease cohort.

These are not donor, marker, or gene-shuffle controls. Those would require releasing or rerunning upstream donor/gene inputs that are outside the public row artifact.


In [ ]:
# -----------------------------------------------------------------------------
# Benchmark 4: cohort formula controls for the released Gaia 44-row score table
# -----------------------------------------------------------------------------
# Inputs:
#   cohort-level-bench/baseline/formula_controls_20260605/
#
# What this cell does:
#   1. Recompute the real Gaia strict-44 metric from gaia_rows.
#   2. Recompute 1,000 global ORR-label shuffles.
#   3. Recompute 1,000 within-disease ORR-label shuffles.
#   4. Summarize each null family with the same p05/median/p95/p-value fields
#      used in the checked-in release artifacts.
#   5. Assert the recomputed rows and summaries match the CSVs exactly.
# -----------------------------------------------------------------------------

COHORT_CONTROL_SEED = 20260605
COHORT_CONTROL_ITERATIONS = 1000
COHORT_CONTROL_SCORE = "gaia_predicted_orr_pct"
COHORT_CONTROL_METRICS = ["pearson", "spearman", "auc_above_disease_median"]

cohort_formula_real_rows = read_csv_rows(COHORT_FORMULA_REAL_CSV)
cohort_formula_metric_rows = read_csv_rows(COHORT_FORMULA_METRICS_CSV)
cohort_formula_summary_rows = read_csv_rows(COHORT_FORMULA_SUMMARY_CSV)


def one_row(rows, **criteria):
    """Return exactly one row matching the supplied CSV key/value criteria."""
    matches = [
        row for row in rows
        if all(str(row.get(key)) == str(value) for key, value in criteria.items())
    ]
    if len(matches) != 1:
        raise AssertionError(f"Expected one row for {criteria}, found {len(matches)}")
    return matches[0]


def shuffle_column_globally(rows, column, rng):
    """Shuffle one column across all rows while leaving every score unchanged."""
    values = [row.get(column, "") for row in rows]
    rng.shuffle(values)
    return [{**row, column: value} for row, value in zip(rows, values)]


def shuffle_column_within_group(rows, column, group_column, rng):
    """Shuffle one column only within disease groups, preserving disease mix."""
    grouped_values = {}
    for row in rows:
        group = str(row.get(group_column, ""))
        grouped_values.setdefault(group, []).append(row.get(column, ""))
    for values in grouped_values.values():
        rng.shuffle(values)

    offsets = {group: 0 for group in grouped_values}
    controlled = []
    for row in rows:
        group = str(row.get(group_column, ""))
        offset = offsets[group]
        offsets[group] += 1
        controlled.append({**row, column: grouped_values[group][offset]})
    return controlled


def summarize_control_values(benchmark_name, control_family, metric, real_value, values):
    """Compute the exact release null-summary fields from per-iteration metrics."""
    clean = [finite_float(value) for value in values]
    clean = [value for value in clean if math.isfinite(value)]
    return {
        "benchmark_level": "cohort",
        "benchmark_name": benchmark_name,
        "control_family": control_family,
        "metric": metric,
        "real_value": real_value,
        "control_n": len(clean),
        "control_mean": mean(clean),
        "control_p05": quantile(clean, 0.05),
        "control_median": quantile(clean, 0.50),
        "control_p95": quantile(clean, 0.95),
        "control_max": max(clean) if clean else math.nan,
        "empirical_p_ge_real": mean([value >= real_value for value in clean]),
    }

# Real metric: this must be identical to the Gaia benchmark computed above.
cohort_formula_real = cohort_metric_row(gaia_rows, COHORT_CONTROL_SCORE, is_orr_scale=True)
expected_real = one_row(
    cohort_formula_real_rows,
    benchmark_name="gaia_44_strict_orr",
    score_col=COHORT_CONTROL_SCORE,
)
for key in ["n_rows", "pearson", "spearman", "auc_above_disease_median"]:
    assert_close(cohort_formula_real[key], expected_real[key], f"cohort_formula_real.{key}")
assert_close(cohort_formula_real["pearson"], gaia_metric["pearson"], "formula real equals Gaia Pearson")

# Null metrics: use one RNG and loop order matching the release script so every
# per-iteration metric row is reproducible from the public Gaia 44-row table.
rng = random.Random(COHORT_CONTROL_SEED)
cohort_recomputed_control_rows = []
for control_family in ["label_shuffle_global", "label_shuffle_within_disease"]:
    for iteration in range(COHORT_CONTROL_ITERATIONS):
        if control_family == "label_shuffle_global":
            controlled_rows = shuffle_column_globally(gaia_rows, "observed_orr_pct", rng)
        else:
            controlled_rows = shuffle_column_within_group(
                gaia_rows,
                "observed_orr_pct",
                "cohort",
                rng,
            )
        metric = cohort_metric_row(controlled_rows, COHORT_CONTROL_SCORE, is_orr_scale=True)
        cohort_recomputed_control_rows.append({
            "benchmark_level": "cohort",
            "benchmark_name": "gaia_44_strict_orr",
            "control_family": control_family,
            "control_iter": iteration,
            "score_col": COHORT_CONTROL_SCORE,
            "n_rows": metric["n_rows"],
            "pearson": metric["pearson"],
            "spearman": metric["spearman"],
            "auc_above_disease_median": metric["auc_above_disease_median"],
        })

# Validate every generated null row against the checked-in per-iteration CSV.
expected_control_by_key = {
    (row["control_family"], int(row["control_iter"])): row
    for row in cohort_formula_metric_rows
}
assert len(cohort_recomputed_control_rows) == len(cohort_formula_metric_rows) == 2000
for row in cohort_recomputed_control_rows:
    expected = expected_control_by_key[(row["control_family"], row["control_iter"])]
    for key in ["n_rows", "pearson", "spearman", "auc_above_disease_median"]:
        assert_close(row[key], expected[key], f"cohort_formula.{row['control_family']}.{row['control_iter']}.{key}")

# Validate null summaries. Empirical p is the fraction of shuffled metrics that
# are greater than or equal to the real, unshuffled metric.
cohort_formula_recomputed_summaries = []
for control_family in ["label_shuffle_global", "label_shuffle_within_disease"]:
    family_rows = [row for row in cohort_recomputed_control_rows if row["control_family"] == control_family]
    for metric in COHORT_CONTROL_METRICS:
        summary = summarize_control_values(
            "gaia_44_strict_orr",
            control_family,
            metric,
            cohort_formula_real[metric],
            [row[metric] for row in family_rows],
        )
        expected = one_row(
            cohort_formula_summary_rows,
            benchmark_name="gaia_44_strict_orr",
            control_family=control_family,
            metric=metric,
        )
        for key in [
            "real_value", "control_n", "control_mean", "control_p05",
            "control_median", "control_p95", "control_max", "empirical_p_ge_real",
        ]:
            assert_close(summary[key], expected[key], f"cohort_formula_summary.{control_family}.{metric}.{key}")
        cohort_formula_recomputed_summaries.append(summary)

cohort_formula_summary_by_key = {
    (row["control_family"], row["metric"]): row
    for row in cohort_formula_recomputed_summaries
}
cohort_formula_control_headline = {
    "real_pearson": cohort_formula_real["pearson"],
    "real_spearman": cohort_formula_real["spearman"],
    "real_auc_above_disease_median": cohort_formula_real["auc_above_disease_median"],
    "global_pearson_p95": cohort_formula_summary_by_key[("label_shuffle_global", "pearson")]["control_p95"],
    "global_pearson_empirical_p_ge_real": cohort_formula_summary_by_key[("label_shuffle_global", "pearson")]["empirical_p_ge_real"],
    "within_disease_pearson_p95": cohort_formula_summary_by_key[("label_shuffle_within_disease", "pearson")]["control_p95"],
    "within_disease_spearman_p95": cohort_formula_summary_by_key[("label_shuffle_within_disease", "spearman")]["control_p95"],
    "within_disease_auc_p95": cohort_formula_summary_by_key[("label_shuffle_within_disease", "auc_above_disease_median")]["control_p95"],
    "within_disease_auc_empirical_p_ge_real": cohort_formula_summary_by_key[("label_shuffle_within_disease", "auc_above_disease_median")]["empirical_p_ge_real"],
}

print(json.dumps(cohort_formula_control_headline, indent=2, sort_keys=True))


## Headline Numbers

These are the cohort numbers we present publicly. They are derived above from row-level artifacts.

In [ ]:
# -----------------------------------------------------------------------------
# Cohort headline table
# -----------------------------------------------------------------------------
# These are the numbers we quote in the README/blog. They are pulled from the
# recomputed metric dictionaries above, not read from a precomputed summary.
# -----------------------------------------------------------------------------

cohort_headlines = {
    "Gaia strict ORR pairs": gaia_metric["n_rows"],
    "Gaia Pearson r": round(gaia_metric["pearson"], 3),
    "Gaia Spearman rho": round(gaia_metric["spearman"], 3),
    "Gaia MAE pp": round(gaia_metric["mae_orr_pct"], 1),
    "Gaia AUC above disease median": round(gaia_metric["auc_above_disease_median"], 3),
    "Atlas Pearson r": round(atlas_metric["pearson"], 3),
    "Atlas Spearman rho": round(atlas_metric["spearman"], 3),
    "Atlas MAE pp": round(atlas_metric["mae_orr_pct"], 1),
    "DepMap covered rows": depmap_metric["n_rows"],
    "DepMap Pearson r": round(depmap_metric["pearson"], 3),
    "DepMap Spearman rho": round(depmap_metric["spearman"], 3),
    "Control global-label Pearson p95": round(cohort_formula_control_headline["global_pearson_p95"], 3),
    "Control within-disease Pearson p95": round(cohort_formula_control_headline["within_disease_pearson_p95"], 3),
    "Control within-disease AUC p95": round(cohort_formula_control_headline["within_disease_auc_p95"], 3),
    "Control within-disease AUC p>=real": round(cohort_formula_control_headline["within_disease_auc_empirical_p_ge_real"], 3),
}
cohort_headlines


## Figure Code

The next cells generate the cohort figures from the same row-level data. These are intentionally simple SVGs so the figure generation is visible and reproducible without extra plotting packages.

In [ ]:
# -----------------------------------------------------------------------------
# Cohort figure helpers
# -----------------------------------------------------------------------------
# The plotting code below is plain SVG generation. It avoids extra plotting
# dependencies and makes every plotted coordinate traceable to the row-level
# benchmark artifacts loaded above.
# -----------------------------------------------------------------------------

def write_svg(path, body, width=760, height=520):
    """Write a complete SVG file from an SVG body string."""
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">{body}</svg>'
    )
    path.write_text(svg + "\\n", encoding="utf-8")
    return path


def show_svg(path):
    """Display SVG inline in Jupyter, or print the path in plain Python."""
    if SVG is None:
        print(path)
    else:
        display(SVG(filename=str(path)))


PALETTE = {
    "breast": "#2f6f9f",
    "crc": "#8d5a2b",
    "nsclc": "#65743a",
    "ovarian": "#8c4f7d",
    "pdac": "#b84a3a",
}


def scatter_observed_vs_predicted(rows, score_col, title, output_name):
    """Plot observed ORR on x-axis and model/baseline predicted ORR on y-axis."""
    width, height = 760, 560
    left, right, top, bottom = 88, 32, 62, 82
    plot_w = width - left - right
    plot_h = height - top - bottom

    # Collect finite points and determine an axis limit that includes all values.
    points = []
    max_value = 0.0
    for row in rows:
        x = finite_float(row.get("observed_orr_pct") or row.get("orr_pct"))
        y = finite_float(row.get(score_col))
        if math.isfinite(x) and math.isfinite(y):
            disease = str(row.get("cohort") or row.get("disease"))
            points.append((x, y, disease, row))
            max_value = max(max_value, x, y)
    axis_max = max(50, math.ceil(max_value / 10) * 10)

    def sx(value):
        return left + plot_w * value / axis_max

    def sy(value):
        return top + plot_h * (1 - value / axis_max)

    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" '
        f'font-weight="700" fill="#202124">{html.escape(title)}</text>',
        f'<line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" '
        f'y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{sx(0)}" y1="{sy(0)}" x2="{sx(axis_max)}" y2="{sy(axis_max)}" '
        'stroke="#9aa0a6" stroke-dasharray="5 5"/>',
    ]

    # Draw grid and tick labels every 10 ORR percentage points.
    for tick in range(0, int(axis_max) + 1, 10):
        x = sx(tick)
        y = sy(tick)
        body.append(f'<line x1="{x:.1f}" y1="{top}" x2="{x:.1f}" y2="{top + plot_h}" stroke="#eef0f2"/>')
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{x:.1f}" y="{top + plot_h + 24}" text-anchor="middle" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick}</text>')
        body.append(f'<text x="{left - 12}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick}</text>')

    # Draw one point per cohort-drug row.
    for x, y, disease, row in points:
        color = PALETTE.get(disease, "#5f6368")
        label = f'{row.get("drug", "")} {disease}: observed {x:.1f}, predicted {y:.1f}'
        body.append(
            f'<circle cx="{sx(x):.1f}" cy="{sy(y):.1f}" r="6" fill="{color}" fill-opacity="0.86">'
            f'<title>{html.escape(label)}</title></circle>'
        )

    # Axis labels and legend.
    body.append(f'<text x="{left + plot_w / 2}" y="{height - 26}" text-anchor="middle" font-family="Inter, Arial" font-size="14" fill="#333">Observed ORR (%)</text>')
    body.append(f'<text transform="translate(24 {top + plot_h / 2}) rotate(-90)" text-anchor="middle" font-family="Inter, Arial" font-size="14" fill="#333">Predicted ORR (%)</text>')
    legend_x = left + plot_w - 150
    for i, disease in enumerate(PALETTE):
        y = top + 18 + i * 20
        body.append(f'<circle cx="{legend_x}" cy="{y}" r="5" fill="{PALETTE[disease]}"/>')
        body.append(f'<text x="{legend_x + 12}" y="{y + 4}" font-family="Inter, Arial" font-size="12" fill="#333">{disease}</text>')

    return write_svg(FIG_DIR / output_name, "".join(body), width, height)


def metric_comparison_bar(methods, output_name):
    """Plot Pearson, Spearman, and disease-median AUC for each cohort method."""
    width, height = 760, 500
    left, right, top, bottom = 82, 28, 58, 78
    plot_w = width - left - right
    plot_h = height - top - bottom
    y_min, y_max = -0.10, 0.80
    metrics = [
        ("pearson", "Pearson r", "#2f6f9f"),
        ("spearman", "Spearman rho", "#b84a3a"),
        ("auc", "AUC", "#65743a"),
    ]

    def sy(value):
        return top + plot_h * (1 - (value - y_min) / (y_max - y_min))

    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">Cohort Metric Comparison</text>',
        f'<line x1="{left}" y1="{sy(0):.1f}" x2="{left + plot_w}" y2="{sy(0):.1f}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]

    # Draw horizontal reference lines.
    for tick in [-0.1, 0.0, 0.2, 0.4, 0.6, 0.8]:
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.1f}</text>')

    # Draw grouped bars by method.
    group_w = plot_w / len(methods)
    bar_w = group_w / 5
    for i, method in enumerate(methods):
        center = left + group_w * (i + 0.5)
        for j, (key, _, color) in enumerate(metrics):
            value = finite_float(method[key])
            x = center + (j - 1) * bar_w * 1.25 - bar_w / 2
            y0 = sy(0)
            y1 = sy(value)
            h = abs(y0 - y1)
            y = min(y0, y1)
            body.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{h:.1f}" fill="{color}"/>')
            body.append(f'<text x="{x + bar_w / 2:.1f}" y="{y - 5:.1f}" text-anchor="middle" font-family="Inter, Arial" font-size="11" fill="#333">{value:.2f}</text>')
        body.append(f'<text x="{center:.1f}" y="{height - 34}" text-anchor="middle" font-family="Inter, Arial" font-size="13" fill="#333">{html.escape(method["label"])}</text>')

    # Legend.
    for j, (_, label, color) in enumerate(metrics):
        x = left + j * 150
        body.append(f'<rect x="{x}" y="{height - 18}" width="12" height="12" fill="{color}"/>')
        body.append(f'<text x="{x + 18}" y="{height - 8}" font-family="Inter, Arial" font-size="12" fill="#333">{label}</text>')

    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

def cohort_formula_control_bar(rows, output_name):
    """Plot the real Gaia Pearson against formula-control null p95 values."""
    width, height = 760, 430
    left, right, top, bottom = 80, 30, 58, 86
    plot_w = width - left - right
    plot_h = height - top - bottom
    y_max = max(0.75, max(row["value"] for row in rows) * 1.12)

    def sy(value):
        return top + plot_h * (1 - value / y_max)

    body = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{left}" y="32" font-family="Inter, Arial" font-size="22" font-weight="700" fill="#202124">Cohort Formula Controls</text>',
        f'<line x1="{left}" y1="{top + plot_h}" x2="{left + plot_w}" y2="{top + plot_h}" stroke="#333"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top + plot_h}" stroke="#333"/>',
    ]
    for tick_i in range(6):
        tick = y_max * tick_i / 5
        y = sy(tick)
        body.append(f'<line x1="{left}" y1="{y:.1f}" x2="{left + plot_w}" y2="{y:.1f}" stroke="#eef0f2"/>')
        body.append(f'<text x="{left - 10}" y="{y + 4:.1f}" text-anchor="end" font-family="Inter, Arial" font-size="12" fill="#5f6368">{tick:.2f}</text>')
    bar_w = plot_w / max(1, len(rows)) * 0.44
    for i, row in enumerate(rows):
        value = row["value"]
        x = left + plot_w * (i + 0.5) / len(rows) - bar_w / 2
        y = sy(value)
        body.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{top + plot_h - y:.1f}" fill="{row["color"]}">')
        body.append(f'<title>{html.escape(row["label"])}: {value:.3f}</title></rect>')
        body.append(f'<text x="{x + bar_w / 2:.1f}" y="{height - 52}" text-anchor="middle" font-family="Inter, Arial" font-size="12" fill="#333">{html.escape(row["label"])}</text>')
    body.append(f'<text x="{left}" y="{height - 18}" font-family="Inter, Arial" font-size="12" fill="#5f6368">Controls use the same released Gaia score and metric path, with ORR labels shuffled.</text>')
    return write_svg(FIG_DIR / output_name, "".join(body), width, height)

print("Figure helpers loaded.")


In [ ]:
# -----------------------------------------------------------------------------
# Generate cohort figures
# -----------------------------------------------------------------------------
# Figures written here are reproducible notebook outputs. They are intentionally
# written under artifacts/notebook_figures/, which is gitignored.
# -----------------------------------------------------------------------------

# Prepare Atlas rows for plotting from the recomputed score.
for row in atlas_rows:
    row["atlas_recomputed_fixed_k8"] = row["atlas_recomputed_fixed_k8"]

# Generate every cohort figure used by the release notebook.
figures = []
figures.append(scatter_observed_vs_predicted(
    gaia_rows,
    "gaia_predicted_orr_pct",
    "Strict 44 Cohorts: Observed vs Gaia Predicted ORR",
    "cohort_gaia_observed_vs_predicted.svg",
))
figures.append(scatter_observed_vs_predicted(
    atlas_rows,
    "atlas_recomputed_fixed_k8",
    "Strict 44 Cohorts: Observed vs Atlas Prior ORR",
    "cohort_atlas_observed_vs_predicted.svg",
))
figures.append(metric_comparison_bar(
    [
        {
            "label": "Gaia",
            "pearson": gaia_metric["pearson"],
            "spearman": gaia_metric["spearman"],
            "auc": gaia_metric["auc_above_disease_median"],
        },
        {
            "label": "Atlas",
            "pearson": atlas_metric["pearson"],
            "spearman": atlas_metric["spearman"],
            "auc": atlas_metric["auc_above_disease_median"],
        },
        {
            "label": "DepMap",
            "pearson": depmap_metric["pearson"],
            "spearman": depmap_metric["spearman"],
            "auc": depmap_metric["auc_above_disease_median"],
        },
    ],
    "cohort_metric_comparison.svg",
))
figures.append(cohort_formula_control_bar(
    [
        {"label": "Real Pearson", "value": cohort_formula_control_headline["real_pearson"], "color": "#2f6f9f"},
        {"label": "Within p95", "value": cohort_formula_control_headline["within_disease_pearson_p95"], "color": "#8c4f7d"},
        {"label": "Global p95", "value": cohort_formula_control_headline["global_pearson_p95"], "color": "#b84a3a"},
    ],
    "cohort_formula_controls.svg",
))

for figure in figures:
    show_svg(figure)

[str(figure.relative_to(ROOT)) for figure in figures]


## Optional External Raw-Data Regeneration

The public notebook above recomputes from checked-in row-level artifacts. To regenerate Atlas or DepMap row-level artifacts from raw external inputs, set the environment variables below and run this cell.

This is optional because raw Atlas and DepMap matrices are not part of the open repo.

In [ ]:
# -----------------------------------------------------------------------------
# Optional raw-source regeneration commands
# -----------------------------------------------------------------------------
# The public benchmark can be verified from checked-in row-level artifacts.
# If a reviewer has raw Atlas or DepMap files locally, this cell reruns the
# baseline scripts that originally generated those row-level artifacts.
# -----------------------------------------------------------------------------

# Optional external regeneration is off by default.
# Set these environment variables if you have the raw source artifacts locally:
#
#   OPEN_BENCHMARKS_ATLAS_CSV=/path/to/ctgov_phase2_solid_tumor_atlas_cohorts_pubmed_supplement.csv
#   OPEN_BENCHMARKS_DEPMAP_DRUG_DIR=/path/to/depmap/drug
#   OPEN_BENCHMARKS_DEPMAP_MODEL_CSV=/path/to/depmap/Model.csv
#
# The commands below run the same public baseline scripts used to create the
# checked-in Atlas and DepMap row-level artifacts.

external_output = ROOT / "artifacts/notebook_external_recompute"
external_output.mkdir(parents=True, exist_ok=True)

atlas_csv = os.environ.get("OPEN_BENCHMARKS_ATLAS_CSV")
depmap_drug_dir = os.environ.get("OPEN_BENCHMARKS_DEPMAP_DRUG_DIR")
depmap_model_csv = os.environ.get("OPEN_BENCHMARKS_DEPMAP_MODEL_CSV")

if atlas_csv:
    atlas_cmd = [
        sys.executable,
        str(ROOT / "cohort-level-bench/baseline/atlas_orr_baseline.py"),
        "--atlas-csv", atlas_csv,
        "--cohort-predictions", str(GAIA_ROWS_CSV),
        "--output-dir", str(external_output / "atlas_orr_baseline"),
        "--surface-score-column", "gaia_predicted_orr_pct",
        "--strict-release-cleaning",
    ]
    print("Running:", " ".join(atlas_cmd))
    subprocess.run(atlas_cmd, check=True)
else:
    print("Skipping raw Atlas regeneration: OPEN_BENCHMARKS_ATLAS_CSV is not set.")

if depmap_drug_dir and depmap_model_csv:
    depmap_cmd = [
        sys.executable,
        str(ROOT / "cohort-level-bench/baseline/depmap_orr_baseline.py"),
        "--depmap-drug-dir", depmap_drug_dir,
        "--model-csv", depmap_model_csv,
        "--cohort-predictions", str(GAIA_ROWS_CSV),
        "--output-dir", str(external_output / "depmap_orr_baseline"),
        "--surface-score-column", "gaia_predicted_orr_pct",
    ]
    print("Running:", " ".join(depmap_cmd))
    subprocess.run(depmap_cmd, check=True)
else:
    print("Skipping raw DepMap regeneration: set OPEN_BENCHMARKS_DEPMAP_DRUG_DIR and OPEN_BENCHMARKS_DEPMAP_MODEL_CSV.")